## Transform Circuits Data

1. Read bronze_circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
4. Rename columns to make them more meaningful (lat → latitude, long → longitude)
5. Filter out rows where circuit_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver_circuits table

### Step 1- Read bronze_circuits table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


### Step 1- Read bronze_circuits table

In [0]:
circuit_df = spark.read.table(bronze_table)

  

In [0]:
display(circuit_df)


#### Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
circuit_selected_df = circuit_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("timestamp"),
    F.col("source_file")
)

In [0]:
display(circuit_selected_df)


#### Standardise column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
#### Rename columns to make them more meaningful (lat → latitude, long → longitude)

In [0]:
circuits_remaned_df = (
    circuit_selected_df
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("circuitName", "circuit_name")                
        .withColumnRenamed("lat", "latitude")
        .withColumnRenamed("long", "longitude")
)

In [0]:
display(circuits_remaned_df)


In [0]:
circuits_remaned_df = (
    circuit_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

- Filter out rows where circuit_id is null (business key validation)


In [0]:
# SQL Expression 
# circuits_valid_df = (
#    circuits_remaned_df
#        .filter("circuit_id is not null")
#)

# Column Expression
circuits_valid_df = (
    circuits_remaned_df
       .filter(F.col("circuit_id").isNotNull()          
        )
)


In [0]:
display(circuits_valid_df)

- Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.distinct()


In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)


### Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
        .withColumn("locality", F.initcap(F.col("locality")))        
)


In [0]:
display(circuits_final_df)


### Write a data to silver_table

In [0]:
(
    circuits_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))